# ZAC 2026 LifeMate — IFM K2 Horizon capability test (Kaggle, CPU)

Runs the same agent as the Nemotron / DeepSeek pipelines (`api_agent/solve.py`) with **K2 Horizon**
(`IFM/K2-Horizon-375B-A23B`, api.ifm.ai, OpenAI format, reasoning always on at "high") on the **634 public-test samples**,
and writes the submission. Every sample gets its own `ToolEnv`; the final request is shown as two transcripts
(Gemini + Gipformer). Train gold is used *inside the prompt* (5 most similar train rows with their gold calls, argument
habits); no public-test text is used as an example (R-RES-3). Reference only: an API model can never be the final
solution (F-1).

**Limit.** IFM gives **10M tokens per key per day** (prompt + completion, reset 00:00 UTC; replayed reasoning counts as
prompt). Estimate ~25–35k tokens per sample → the 634 samples need **~20M tokens, i.e. two keys or two days**. With
**several keys** in the secret, the run moves to the next key as soon as one hits its cap (or is rejected) and stops only
when the last one does; everything is saved, resume later (below). No prices are published.

**One-time setup**
1. Run `py kaggle/make_k2.py` on your PC and upload `colab/zola_deepseek_bundle.zip` as a **private** Kaggle dataset
   (competition data, R-RES-4; re-upload it if you had an older one: cell 1 refuses a solve.py without key rotation).
   *Add Input* → attach it.
2. *Add-ons → Secrets* → add `IFM_API_KEY` and attach it to this notebook. Several keys go into this one secret,
   separated by commas or new lines; they are used in that order.
3. *Settings → Internet on*. *Accelerator: None.*

**How to run**
- Quick test in the editor: `LIMIT = 3` in cell 1, run cells 1–3 (separate files).
- Full run: `LIMIT = None`, then *Save Version → Save & Run All (Commit)*.
- **Resume** (daily cap, API errors, killed session): *Add Input → Notebook Outputs* → this notebook's last version, then
  *Save & Run All* again. Cell 1 copies the checkpoints; finished samples are never sent again.

**Outputs** (`/kaggle/working`): `submission.json` (= `submission_k2.jsonl`, JSON Lines as the competition requires),
`traces_public_k2.jsonl` (per sample: every step's thought, text, tool calls with status and result),
`cache_public_k2.jsonl`, `run_k2.log`. Samples K2 has not solved yet are taken from the 68.42 Nemotron submission
(`FILL_FROM_BASE`), so a partial run is still a valid file — cell 3 says how many; the LB score measures K2 only once
that number is 0. Public traces are reference only: never use them for training or prompts (F-2).

In [ ]:
# 1. Settings, files, API key
import getpass, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path
BACKEND = "ifm"       # IFM K2 Horizon (IFM/K2-Horizon-375B-A23B)
WORKERS = 4           # samples at the same time (IFM's per-minute limits are unpublished; lower it on many 429s)
LIMIT = None          # e.g. 3 for a quick test (separate files); None = all 634 public samples
FILL_FROM_BASE = True # samples not solved yet (daily cap, errors) are taken from the 68.42 Nemotron submission
TRAIN_ROWS = 0        # optional train check (cell 4) on this many rows with gold text; uses the same daily token cap

COLAB = False
IN = Path(os.environ.get("ZOLA_IN", "/kaggle/input"))
RUN = Path(os.environ.get("ZOLA_OUT", "/kaggle/working"))  # the version's Output tab
ROOT = Path(os.environ.get("ZOLA_ROOT", "/tmp/ZOLA"))     # writable copy in the layout solve.py expects
if not (ROOT / "api_agent/solve.py").is_file():
    src = IN
    z = next(IN.rglob("zola_deepseek_bundle.zip"), None)  # Kaggle normally unpacks an uploaded zip; handle both
    if z:
        src = ROOT.with_name(ROOT.name + "_src")
        zipfile.ZipFile(z).extractall(src)
    # found by content, not folder name, in case Kaggle renames folders with spaces or '+'
    solve = next((p for p in src.rglob("solve.py") if (p.parent / "transcripts_gemini_public_test.json").is_file()), None)
    tj, pj = next(src.rglob("train.jsonl"), None), next(src.rglob("public_test.jsonl"), None)
    if not (solve and tj and pj):
        raise SystemExit("Bundle not found: attach the zola_deepseek_bundle dataset to this notebook (Add Input).")
    shutil.rmtree(ROOT, ignore_errors=True)
    skip = shutil.ignore_patterns("__pycache__")
    shutil.copytree(solve.parent, ROOT / "api_agent", ignore=skip)
    shutil.copytree(tj.parent, ROOT / "train+public_test/train", ignore=skip)
    shutil.copytree(pj.parent, ROOT / "train+public_test/public_test", ignore=skip)

RUN.mkdir(parents=True, exist_ok=True)
for p in IN.rglob("*k2*.jsonl"):  # resume: an earlier version's output attached as input
    q = RUN / p.name
    if not q.is_file() or sum(1 for _ in open(p, encoding="utf-8")) > sum(1 for _ in open(q, encoding="utf-8")):
        shutil.copy(p, q)
        print("checkpoint copied from", p)
old = [float(p.read_text()) for p in IN.rglob("spend_usd.txt")]  # keep the budget across versions
if old and max(old) > (float((RUN / "spend_usd.txt").read_text()) if (RUN / "spend_usd.txt").is_file() else 0.0):
    (RUN / "spend_usd.txt").write_text(f"{max(old):.6f}")

if not os.environ.get("IFM_API_KEY"):  # one secret; several keys separated by commas or new lines
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["IFM_API_KEY"] = UserSecretsClient().get_secret("IFM_API_KEY").strip()
    except Exception as e:
        raise SystemExit(f"No key ({type(e).__name__}): Add-ons -> Secrets -> IFM_API_KEY, attach it to this notebook.")
import requests
try:
    requests.head("https://api.ifm.ai", timeout=15)
except requests.RequestException as e:
    raise SystemExit(f"No internet ({type(e).__name__}): turn on Settings -> Internet, then run again.")
RUN.mkdir(parents=True, exist_ok=True)
CODE = ROOT / "api_agent"
if 'KEY_AT' not in (CODE / "solve.py").read_text(encoding="utf-8"):
    raise SystemExit("solve.py is too old (no ifm key rotation): rebuild the bundle with kaggle/make_k2.py and upload it again.")
SPEND, PAUSE = RUN / "spend_usd.txt", RUN / "PAUSE"
COMMON = ["--backend", BACKEND]
r = subprocess.run([sys.executable, "solve.py", "--split", "train", *COMMON, "--self-check"], cwd=CODE,
                   capture_output=True, text=True)
print("solve.py self-check:", (r.stdout + r.stderr).strip()[-300:])
assert r.returncode == 0, "solve.py self-check failed"
spent = float(SPEND.read_text()) if SPEND.is_file() else 0.0
print("run folder:", RUN, "| IFM keys:", len(os.environ["IFM_API_KEY"].replace(",", " ").split()))
import signal, time


def run_pass(cmd, log_name):
    """Runs solve.py, mirroring its output to the log. Returns the line that says why it stopped early, or None."""
    halt = None
    with open(RUN / log_name, "a", encoding="utf-8") as log:
        p = subprocess.Popen(cmd, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             encoding="utf-8", env={**os.environ, "PYTHONIOENCODING": "utf-8"})

        def drain():
            nonlocal halt
            for line in p.stdout:
                print(line, end="")
                log.write(line)
                log.flush()
                if line.startswith("STOPPED:"):  # daily token cap, PAUSE file or rejected key
                    halt = line.strip()
            return p.wait()
        try:
            rc = drain()
        except KeyboardInterrupt:
            PAUSE.touch()
            print("\npausing: running samples finish and are saved (Stop again to abort them)...", flush=True)
            try:
                drain()
            except KeyboardInterrupt:
                p.send_signal(signal.SIGINT)
                time.sleep(3)
                p.kill()
            return "paused by hand"
    return halt or (f"solve.py exited with {rc}" if rc else None)

In [ ]:
# 2. Public run. Every finished sample is saved at once; rerun this cell to resume (finished ones are not sent again).
#    Pause: press Stop once (running samples finish), or create PAUSE in the run folder. Stop twice = abort.
PAUSE.unlink(missing_ok=True)
tag = f"_test{LIMIT}" if LIMIT else ""
SUB, TRACES = RUN / f"submission_k2{tag}.jsonl", RUN / f"traces_public_k2{tag}.jsonl"
cmd = [sys.executable, "-u", "solve.py", "--split", "public_test", *COMMON,
       "--transcripts", str(CODE / "transcripts_gemini_public_test.json"),
       "--extra-transcripts", str(CODE / "transcripts_gipformer_public_test.json"),
       "--workers", str(WORKERS), "--cache", str(RUN / f"cache_public_k2{tag}.jsonl"), "--traces", str(TRACES),
       "--out", str(SUB), "--stop-file", str(PAUSE)]
cmd += ["--base-submission", str(CODE / "submission_68.42.jsonl")] if FILL_FROM_BASE else []
cmd += ["--limit", str(LIMIT)] if LIMIT else []


for n in (1, 2):  # pass 2 only retries samples that failed in pass 1 (API errors)
    print(f"=== pass {n} ===", flush=True)
    halt = run_pass(cmd, f"run_k2{tag}.log")
    if halt:
        print("\nHALTED:", halt, "\nEverything finished so far is saved; rerun this cell to resume.")
        break

In [ ]:
# 3. Validate the submission, summarise the run, export submission.json
import collections
r = subprocess.run([sys.executable, "-m", "zaic_tools", "--split", "public_test", "--root", "..", "--validate", str(SUB)],
                   cwd=ROOT / "train+public_test/public_test", capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
shutil.copy(SUB, RUN / "submission.json")  # same JSON Lines content: one line per sample (R-SUB-1)
rows = [json.loads(l) for l in open(TRACES, encoding="utf-8")] if TRACES.is_file() else []
n_lines = sum(1 for _ in open(SUB, encoding="utf-8"))
print(f"K2 Horizon solved {len(rows)} samples; {n_lines - len(rows)} of {n_lines} lines are from the 68.42 base submission")
if rows:
    print("stops:", dict(collections.Counter(o["stop"] for o in rows)))
    print(f"steps/sample {sum(len(o['steps']) for o in rows) / len(rows):.2f}, "
          f"seconds/sample {sum(o['seconds'] for o in rows) / len(rows):.1f}")
print("files:", RUN / "submission.json", SUB, TRACES, sep="\n  ")

In [ ]:
# 4. Optional: train check on TRAIN_ROWS rows with gold text, scored with the official formula.
#    Examples and argument habits for these rows come only from the other train rows.
ids = [json.loads(l)["id"] for l in open(ROOT / "train+public_test/train/train.jsonl", encoding="utf-8") if l.strip()]
pick = ids[::max(1, len(ids) // max(1, TRAIN_ROWS))][:TRAIN_ROWS]
PAUSE.unlink(missing_ok=True)
halt = None if not pick else run_pass([sys.executable, "-u", "solve.py", "--split", "train", *COMMON, "--ids", ",".join(pick),
                 "--workers", str(WORKERS), "--cache", str(RUN / f"cache_train_k2{len(pick)}.jsonl"),
                 "--traces", str(RUN / f"traces_train_k2{len(pick)}.jsonl"), "--stop-file", str(PAUSE)],
                f"run_train_k2{len(pick)}.log")
print("HALTED:", halt) if halt else None

In [ ]:
# Status (any time, needs only cell 1)
for p in sorted(RUN.glob("cache_*.jsonl")):
    print(f"{p.name}: {sum(1 for _ in open(p, encoding='utf-8'))} samples")
print("paused:", PAUSE.exists())